# 02b - Sampling robustness check (M2-14)

Before 2025 each tag-month is a **sample**: one random weekday and one random weekend day (seed 42, "Option A" in `config.yaml`), each question carrying a weight so that monthly figures stand for the whole month. From January 2025 every question is collected, and tag-months under 300 questions are collected in full. This notebook measures how much the sampling can move the numbers used in the analysis, in three ways: estimated against complete monthly volume, weighted against unweighted resolution rates, and the same rates under a second random seed.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src import robustness as rb
pd.set_option("display.width", 140, "display.max_columns", 30)
v1 = pd.read_parquet(ROOT / "data/processed/questions_clean_v1.parquet",
                     columns=["question_id", "collected_tags", "created_at", "sampling_weight", "target_resolved_7d", "target_answered_7d"])
complete = pd.read_csv(ROOT / "data/processed/monthly_counts.csv")
len(v1)

302026

## 1. Estimated against complete monthly volume

`estimated` is the sum of sampling weights in a tag-month; `complete` is Member 1's monthly count (`filter=total`).

In [2]:
check = rb.volume_check(v1, complete)
pd.Series(rb.volume_summary(check)).round(2).to_frame("value")

,value
sampled_tag_months,912.00
sampled_mean_abs_err_pct,7.12
sampled_median_abs_err_pct,5.15
sampled_p90_abs_err_pct,16.86
sampled_max_abs_err_pct,67.26
full_tag_months,256.00
full_mean_abs_err_pct,0.95
full_max_abs_err_pct,43.24


In [3]:
rb.yearly_volume_bias(check).round(1)

,estimated,complete,pct_err
year,,,
2019,870460.0,867103,0.4
2020,985658.0,986300,-0.1
2021,836761.0,834081,0.3
2022,718449.0,728639,-1.4
2023,383027.0,383784,-0.2
2024,169584.0,173334,-2.2
2025,42610.0,43010,-0.9
2026,6469.0,6520,-0.8


**Reading.** Yearly totals are close to the complete counts (within about 2%), so the sampling is not biased overall. Single months are noisier: a typical sampled tag-month is off by about 5%, one in ten by more than 17%, because only two days of each month are drawn. Charts of monthly volume should use the complete weekly counts; the sampled estimate should only be used for rates.

In [4]:
by_tag = check[(check.month < rb.FULL_FROM) & check.estimated.notna() & (check.complete > 0)].groupby("tag").apply(
    lambda g: pd.Series({"mean_abs_err_pct": g.pct_err.abs().mean(), "total_bias_pct": (g.estimated.sum() / g.complete.sum() - 1) * 100}), include_groups=False)
by_tag.round(1)

,mean_abs_err_pct,total_bias_pct
tag,,
android,7.9,-0.1
docker,12.4,-10.1
excel,11.3,-0.1
fastapi,0.0,0.0
java,8.5,1.1
javascript,5.4,-0.6
openai-api,0.0,0.0
pandas,9.1,-0.1
powerbi,0.6,-0.8


**Fully collected months (2025 onward)** should match exactly. The largest gaps are shown below.

In [5]:
late = check[(check.month >= rb.FULL_FROM) & check.estimated.notna() & (check.complete > 0)]
late[late.pct_err.abs() > 3].round(1).sort_values("pct_err")

,tag,month,estimated,complete,pct_err
164,docker,2025-01,189.0,333,-43.2
165,docker,2025-02,186.0,298,-37.6
166,docker,2025-03,186.0,245,-24.1
823,powerbi,2026-04,7.0,8,-12.5
167,docker,2025-04,163.0,182,-10.4
816,powerbi,2025-09,18.0,20,-10.0
730,pandas,2026-03,20.0,22,-9.1
632,openai-api,2025-09,10.0,11,-9.1
1103,reactjs,2026-08,10.0,11,-9.1
1098,reactjs,2026-03,32.0,34,-5.9


**Reading.** Docker in January to March 2025 is 24% to 43% below the complete count. Member 1 traced this to the platform: the API's `filter=total` counts locked or deleted questions that the paged endpoint never returns, so the collected rows are the accessible questions. The few other small gaps (3% to 12%) are on months with only 8 to 120 questions and have the same cause. fastapi and openai-api have no questions before they existed (28 tag-months with a complete count of 0).

## 2. Weighted against unweighted resolution rates

In [6]:
rb.weighted_vs_unweighted(v1).round(3)

,unweighted_resolved,weighted_resolved,unweighted_answered,weighted_answered,resolved_diff_pp,answered_diff_pp
year,,,,,,
2019,0.457,0.457,0.785,0.782,-0.085,-0.296
2020,0.450,0.450,0.778,0.776,-0.076,-0.247
2021,0.432,0.437,0.763,0.764,0.507,0.121
2022,0.407,0.408,0.751,0.752,0.137,0.105
2023,0.353,0.360,0.699,0.707,0.702,0.813
2024,0.344,0.345,0.695,0.690,0.066,-0.491
2025,0.335,0.335,0.702,0.702,0.000,0.000
2026,0.248,0.248,0.749,0.749,0.000,0.000


In [7]:
tm = rb._by_tag(v1)
import numpy as np
def r(g):
    w = g.sampling_weight
    return pd.Series({"gap_pp": abs(np.average(g.target_resolved_7d, weights=w) - g.target_resolved_7d.mean()) * 100, "n": len(g)})
gap = tm.groupby(["tag", "month"]).apply(r, include_groups=False)
gap.gap_pp.describe().round(2).to_frame("|weighted - unweighted| resolved rate, percentage points")

,"|weighted - unweighted| resolved rate, percentage points"
count,1168.00
mean,0.75
std,1.16
min,0.00
25%,0.00
50%,0.29
75%,1.07
max,14.17


**Reading.** By year the weighted and unweighted resolution rates differ by under 1 percentage point, so yearly conclusions do not depend on the weights. For single tag-months the gap averages under 1 point but reaches 14 points in the worst case, so monthly rate charts must use the weighted tables (`monthly_tag_tables.csv`), and months flagged `low_sample` should be greyed out.

## 3. Second random seed

`sql` and `excel` were re-sampled with seed 7 for the sampled years 2019 to 2024 (same rule, different random days), using `collect_tag(tag, seed=7, window_end="2024-12")` from `src/collect_questions.py` with output redirected to a scratch folder so the project files were untouched. Answer dates were fetched for the new questions (about 1,000 API calls, logged). Results are saved in `data/processed/seed_comparison.csv`.

In [8]:
s = pd.read_csv(ROOT / "data/processed/seed_comparison.csv")
w = s.pivot_table(index=["tag", "year"], columns="seed", values=["resolved_7d", "answered_7d"])
out = pd.DataFrame({
    "resolved seed42": w[("resolved_7d", "seed42")], "resolved seed7": w[("resolved_7d", "seed7")],
    "resolved diff (pp)": (w[("resolved_7d", "seed7")] - w[("resolved_7d", "seed42")]) * 100,
    "answered seed42": w[("answered_7d", "seed42")], "answered seed7": w[("answered_7d", "seed7")],
    "answered diff (pp)": (w[("answered_7d", "seed7")] - w[("answered_7d", "seed42")]) * 100,
})
out.round(3)

resolved seed42  resolved seed7  resolved diff (pp)  answered seed42  answered seed7  answered diff (pp)
tag   year                                                                                                          
excel 2019            0.436           0.422              -1.401            0.766           0.785               1.856
      2020            0.397           0.429               3.217            0.747           0.752               0.526
      2021            0.413           0.406              -0.762            0.757           0.723              -3.452
      2022            0.361           0.348              -1.344            0.715           0.748               3.226
      2023            0.370           0.407               3.773            0.749           0.794               4.500
      2024            0.419           0.381              -3.813            0.799           0.760              -3.909
      all             0.402           0.405               0.235            0.752           0.760               0.746
sql   2019            0.525           0.517              -0.833            0.883           0.888               0.542
      2020            0.546           0.548               0.204            0.892           0.897               0.531
      2021            0.496           0.504               0.791            0.873           0.873              -0.010
      2022            0.436           0.455               1.921            0.838           0.840               0.213
      2023            0.430           0.405              -2.539            0.842           0.837              -0.482
      2024            0.407           0.456               4.884            0.817           0.816              -0.066
      all             0.495           0.498               0.309            0.869           0.871               0.251

**Reading.** Over the whole sampled period the two seeds agree closely: the resolved rate differs by about 0.2 to 0.3 percentage points and the answered rate by 0.3 to 0.7 points for both tags. By single year the difference grows to about 3 to 5 points (the smaller `excel` sample and `sql` 2024), the size of the noise from sampling two days per month. So the class balance is stable, and the year-by-year trends are reliable only when the change between years is larger than a few points.

## Conclusions for the limitations section

1. Yearly volume and resolution rates are reliable; the sampling introduces no overall bias.
2. Monthly volume from the sample is noisy (typical error about 5%); use Member 1's complete weekly counts for volume charts.
3. Use weighted rates and grey out `low_sample` months; the weights matter little by year and more by tag-month.
4. Differences between two years below about 3 to 5 points in a small tag should not be read as a real change.
5. Docker (Jan to Mar 2025) and a few small months are short of the complete count because of a platform limitation, not a collection error.